# Imports

In [12]:
import duckdb
import pandas as pd
from dotenv import load_dotenv
import os
import numpy as np

# Connections

In [2]:
load_dotenv(dotenv_path="../.env")
minio_user = os.getenv("MINIO_ROOT_USER", "minioadmin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

In [3]:
con.execute(f"""
    CREATE SECRET IF NOT EXISTS (
        TYPE S3,
        KEY_ID '{minio_user}',
        SECRET '{minio_password}',
        ENDPOINT 'localhost:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")

In [4]:
print("Loading data from Data Lake...")
df = con.execute("SELECT * FROM read_parquet('s3://analytics-data/ml_features.parquet')").df()

Loading data from Data Lake...


In [5]:
print("\n DATASET ")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns (Features): {df.shape[1]}")


 DATASET 
Total Rows: 514,406
Total Columns (Features): 49


In [6]:
print("\n DATA BREAKDOWN ")
print(df.groupby(['asset_class', 'asset_symbol', 'interval']).size().reset_index(name='Row Count'))


 DATA BREAKDOWN 
   asset_class asset_symbol interval  Row Count
0       Crypto     1000PEPE       1d        846
1       Crypto     1000PEPE       1h      24868
2       Crypto          ADA       1d       1622
3       Crypto          ADA       1h      43496
4       Crypto          ADA        W         62
5       Crypto         AVAX       1d       1441
6       Crypto         AVAX       1h      39151
7       Crypto         AVAX        W         36
8       Crypto          BTC       1d       1980
9       Crypto          BTC       1h      52085
10      Crypto          BTC        W        113
11      Crypto         DOGE       1d       1546
12      Crypto         DOGE       1h      41669
13      Crypto         DOGE        W         51
14      Crypto          DOT       1d       1622
15      Crypto          DOT       1h      43494
16      Crypto          DOT        W         62
17      Crypto         DYDX       1d       1415
18      Crypto         DYDX       1h      38529
19      Crypto        

In [7]:
print("Loading Feature Statistics...")
stats_df = con.execute("SELECT * FROM read_parquet('s3://analytics-data/feature_statistics.parquet')").df()

Loading Feature Statistics...


In [8]:
best_features_df = stats_df[stats_df['quality_flag'] == 'PASS'].nlargest(15, 'importance_score')
ml_features_list = best_features_df['feature_name'].tolist()
print("\n ML FEATURE SELECTION ")
print("TOP 15 indicators to train model")
for i, feature in enumerate(ml_features_list, 1):
    print(f"{i}. {feature}")


 ML FEATURE SELECTION 
TOP 15 indicators to train model
1. hl_ratio
2. close_position
3. roc_20
4. bb_percentage
5. stoch_k
6. bb_width
7. rsi_14
8. volume_ratio
9. stoch_d
10. roc_10
11. obv
12. macd_histogram
13. prev_volume
14. macd
15. macd_signal


In [13]:
symbol = 'BTC'
timeframe = '1d'
clean_data = df[(df['asset_symbol'] == symbol) & (df['interval'] == timeframe)].copy()
clean_data = clean_data.dropna(subset=ml_features_list + ['returns_1d'])
X = clean_data[ml_features_list]
y = np.where(clean_data['returns_1d'] > 0, 1, 0)

In [ ]:
print(f" Separated Data for {symbol} [{timeframe}] ---")
print(f"Total Rows: {len(clean_data)}\n")
display(X.head())

--- Separated Data for BTC [1d] ---
Total Rows: 1980



,hl_ratio,close_position,roc_20,bb_percentage,stoch_k,bb_width,rsi_14,volume_ratio,stoch_d,roc_10,obv,macd_histogram,prev_volume,macd,macd_signal
111522,0.038970,0.576617,3.587793,1.186174,83.014572,8.199981,62.836489,1.915001,89.675056,5.060879,153193.618,70.537667,5558.201,54.498727,-16.038940
111523,0.014155,0.546584,9.281322,1.116429,89.435337,9.421770,64.089183,0.889259,88.864255,7.190651,157514.386,87.351897,9725.902,93.150931,5.799035
111524,0.045821,0.652174,9.732915,1.118380,86.237846,11.141184,66.995193,1.672036,86.229252,9.260398,165847.343,104.171552,4320.768,136.013474,31.841922
111525,0.020743,0.432489,11.642564,0.969848,77.299925,11.073411,63.149303,1.299720,84.324369,8.391044,159295.077,101.336433,8332.957,158.512464,57.176031
111526,0.022288,0.485265,6.491024,0.906019,76.776365,11.828403,62.921444,1.037789,80.104712,7.055128,154133.723,93.279225,6552.266,173.775063,80.495837


In [16]:
print("\nFirst 5 Target Values (1 = UP, 0 = DOWN):")
print(y[:5])


First 5 Target Values (1 = UP, 0 = DOWN):
[1 1 1 0 0]
